In [9]:

%pip install gymnasium

import numpy as np
import matplotlib.pyplot as plt

from datetime import datetime

import multiprocessing
from multiprocessing import Pool

import gymnasium as gym
import sys

# environment
ENV_NAME = 'CartPole-v1'

Note: you may need to restart the kernel to use updated packages.


In [10]:
### neural network

# hyperparameters
env = gym.make(ENV_NAME)
D = np.prod(env.observation_space.shape)
M = 32  # reduced from 128 - CartPole only needs 4 inputs -> 2 outputs
K = env.action_space.n


In [11]:
def relu(x):
  return x * (x > 0)

class ANN:
  def __init__(self, D, M, K, f=relu):
    self.D = D
    self.M = M
    self.K = K
    self.f = f

  def init(self):
    D, M, K = self.D, self.M, self.K
    self.W1 = np.random.randn(D, M) / np.sqrt(D)
    self.b1 = np.zeros(M)
    self.W2 = np.random.randn(M, K) / np.sqrt(M)
    self.b2 = np.zeros(K)

  def forward(self, X):
    Z = self.f(X @ self.W1 + self.b1)
    return Z @ self.W2 + self.b2

  def sample_action(self, x):
    # assume input is a single state of size (D,)
    # first make it (N, D) to fit ML conventions
    X = np.atleast_2d(x)
    Y = self.forward(X)
    return np.argmax(Y[0]) # first row

  def get_params(self):
    # return a flat array of parameters
    return np.concatenate([self.W1.flatten(), self.b1, self.W2.flatten(), self.b2])

  def get_params_dict(self):
    return {
      'W1': self.W1,
      'b1': self.b1,
      'W2': self.W2,
      'b2': self.b2,
    }

  def set_params(self, params):
    # params is a flat list
    # unflatten into individual weights
    D, M, K = self.D, self.M, self.K
    self.W1 = params[:D * M].reshape(D, M)
    self.b1 = params[D * M: D * M + M]
    self.W2 = params[D * M + M: D * M + M + M * K].reshape(M, K)
    self.b2 = params[-K:]

In [12]:
class OnlineStandardScaler:
  def __init__(self, num_inputs):
    self.n = 0
    self.mean = np.zeros(num_inputs)
    self.ssd = np.zeros(num_inputs)

  def partial_fit(self, X):
    self.n += 1
    delta = X - self.mean
    self.mean += delta / self.n
    delta2 = X - self.mean
    self.ssd += delta * delta2

  def transform(self, X):
    m = self.mean
    v = (self.ssd / self.n).clip(min=1e-2)
    s = np.sqrt(v)
    return (X - m) / s


scaler = OnlineStandardScaler(D)

In [13]:
class Optimizer:
  def __init__(self, params, lr, beta1=0.9, beta2=0.999, eps=1e-8):
    self.lr = lr
    self.m = 0 # first moment
    self.v = 0 # second moment
    self.b1 = beta1
    self.b2 = beta2
    self.eps = eps
    self.t = 1 # time step
    self.params = params

#   def update(self, g):
#     # new m
#     self.m = self.b1 * self.m + (1 - self.b1) * g
#     # new v
#     self.v = self.b2 * self.v + (1 - self.b2) * g**2
#     # bias correction
#     m_hat = self.m / (1 - self.b1**self.t)
#     v_hat = self.v / (1 - self.b2**self.t)
#     # update time step
#     self.t += 1
#     # update params
#     self.params += self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
#     return self.params

  def update(self, g):
    # update params
    self.params += self.lr * g
    return self.params

In [14]:
def evolution_strategy(
  f,
  population_size,
  sigma,
  lr,
  initial_params,
  num_iters,
  pool,
  ):

  # assume initial params is a 1-D array
  num_params = len(initial_params)
  reward_per_iteration = np.zeros(num_iters)

  # create optimizer
  params = initial_params
  adam = Optimizer(params, lr)

  for t in range(num_iters):
    t0 = datetime.now()
    eps = np.random.randn(population_size, num_params)

    ### fast way
    params_try = [params + sigma * eps[i] for i in range(population_size)] + \
      [params - sigma * eps[i] for i in range(population_size)]
    R = pool.map(f, params_try)
    R = np.array(R)

    # split into positive and negative rewards
    R_pos = R[:population_size]
    R_neg = R[population_size:]

    m = R.mean()
    s = R.std()
    if s == 0:
      print("sd = 0, m =", m)
      s = 0.1

    reward_per_iteration[t] = m
    g = eps.T @ (R_pos - R_neg) / (population_size * s)
    params = adam.update(g)

    print("Iter:", t, "Avg Reward:", m, "Max Reward:", R.max(), "Duration:", datetime.now() - t0)

    # early stopping: avg reward close to max (500) for 5 iterations
    if t > 5 and np.mean(reward_per_iteration[t - 4: t + 1]) >= 490:
      print("Good performance, quitting early")
      break

  return params, reward_per_iteration[:t+1]  # fixed off-by-one


In [15]:
def reward_function(params, record=False):
  # run one episode of env w/ params
  model = ANN(D, M, K)
  model.set_params(params)

  if record:
    env = gym.make(ENV_NAME, render_mode="rgb_array")
    env = gym.wrappers.RecordVideo(env, video_folder="videos", episode_trigger=lambda eps: True)
  else:
    env = gym.make(ENV_NAME)
  env = gym.wrappers.RecordEpisodeStatistics(env)

  # play one episode and return total reward
  episode_reward = 0
  done = False
  state, _ = env.reset()
  while not done:
    # NOTE: scaler removed - it uses a global that is not shared across
    # multiprocessing workers (each worker gets a stale fork-time copy),
    # so partial_fit updates are lost and normalization is incorrect.
    # CartPole observations are small and bounded, so no scaling is needed.
    action = model.sample_action(state)
    state, reward, done, truncated, info = env.step(action)
    done = done or truncated
    episode_reward += reward

  env.close()
  return episode_reward


In [16]:
if __name__ == '__main__':

  # create model
  model = ANN(D, M, K)

  if len(sys.argv) > 1 and sys.argv[1] == 'play':
    # play with a saved model
    j = np.load('ars_mujoco_results.npz')
    best_params = np.concatenate([j['W1'].flatten(), j['b1'], j['W2'].flatten(), j['b2']])
  else:
    # pool for parallel evaluation
    pool = Pool(4)

    # train and save model
    model.init()
    params = model.get_params()
    best_params, rewards = evolution_strategy(
      f=reward_function,
      population_size=30,
      sigma=0.05,
      lr=0.02,
      initial_params=params,
      num_iters=300,  # reduced from 1000 - CartPole solves in ~50-150 iters
      pool=pool,
    )

    # plot the rewards per iteration
    plt.plot(rewards)
    plt.show()

    # save params
    model.set_params(best_params)
    np.savez(
      'ars_mujoco_results.npz',
      train=rewards,
      **model.get_params_dict(),
    )


Process SpawnPoolWorker-24:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/DL_env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/DL_env/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/anaconda3/envs/DL_env/lib/python3.11/multiprocessing/pool.py", line 114, in worker
    task = get()
           ^^^^^
  File "/opt/anaconda3/envs/DL_env/lib/python3.11/multiprocessing/queues.py", line 367, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'reward_function' on <module '__main__' (built-in)>
Process SpawnPoolWorker-25:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/DL_env/lib/python3.11/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/anaconda3/envs/DL_env/lib/python3.11/multiprocessing/process.py", line 108, in run
    self._t

KeyboardInterrupt: 

In [ ]:
print("Test:", reward_function(best_params, record=True))

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Test: 500.0


In [ ]:
# display the video in jupyter/ipython notebook
from IPython.display import Video

video_path = "/content/videos/rl-video-episode-0.mp4"
Video(video_path)

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open("/content/videos/rl-video-episode-0.mp4",'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

![](https://deeplearningcourses.com/notebooks_v3_pxl?sc=8fP_8gnYambvgyDFSyTaIA&n=ARS+Cartpole)